# Questão 1


Para minimizar uma função, começamos calculando sua primeira derivada e encontrando um dos seus pontos críticos.

$
\begin{aligned}
J(x) = (x - c)^T A (x - c) + b
\end{aligned}
$

Temos que $x^T x = x^2$, portanto podemos dizer que

$
\begin{aligned}
&J(x) = A (x - c)^2 + b\\
&J'(x) = 2A (x - c)\\
\end{aligned}
$

Agora temos que encontrar o valor de x que faz com que $J'(x) = 0$

$
\begin{aligned}
&2A (x - c) = 0\\
&x - c = \frac{0}{2A}\\
&x - c = 0\\
&x = c
\end{aligned}
$

Para garantir que este valor minimiza a função, analisamos a segunda derivada de J

$
\begin{aligned}
&J''(x) = 2A\\
\end{aligned}
$

Sabemos que um ponto crítico de uma função é mínimo se a segunda derivada for maior que zero e temos que A é uma matriz definida positiva definida, portanto 2A deve ser maior que zero. Isso prova que quando x = c temos o mínimo da função.

# Questão 2

In [85]:
import numpy as np
import plotly.graph_objects as go

def J(x, A, b, c):
    # J(x) = (x - c)^T * A * (x - c) + b
    diff = x - c
    return float(diff.T @ A @ diff) + b

def gradiente_J(x, A, c):
    # gradiente_J(x) = 2 * A * (x - c)
    return 2 * (A @ (x - c))

def descida_gradiente(A, b, c, alpha, x0, max_iter=100, tol=1e-5):
    A = np.array(A)
    c = np.array(c)
    x = np.array(x0, dtype=float)

    x_hist = []
    hist = []

    Jx = J(x, A, b, c)
    hist.append(f"Iteração 0 : x = {x} -> J(x) = {Jx:.6f}\n")
    x_hist.append(x)

    for i in range(max_iter):
        grad = gradiente_J(x, A, c)
        xi = x - (alpha * grad)
        Jxi = J(xi, A, b, c)
        Jx = J(x, A, b, c)

        if np.linalg.norm(xi - x) <= tol or abs(Jxi - Jx) <= tol:
            hist.append(f"Iteração {i + 1} : x = {xi} -> J(x) = {Jxi:.6f} [Convergiu]\n")
            x_hist.append(xi)
            return xi, hist, x_hist

        x = xi
        hist.append(f"Iteração {i + 1} : x = {x} -> J(x) = {Jxi:.6f}\n")
        x_hist.append(xi)

    return x, hist, x_hist

A = np.array([[10, 0], [0, 10]])
b = 0
c = np.array([5, 5])
x0 = np.array([4, 6])
alpha = 0.05

x_opt, hist, x_hist = descida_gradiente(A, b, c, alpha, x0)

print("".join(hist))
print(f"Ponto de mínimo encontrado: {x_opt}")

alphas = [0.025, 0.05, 0.075]
cores = ["#1f77b4", "#ff7f0e", "#a31313"]
resultados_alpha = {}

for a in alphas:
    x_opt_i, hist_i, x_hist_i = descida_gradiente(A, b, c, a, x0)
    resultados_alpha[a] = np.array(x_hist_i)
    print(f"alpha = {a:<6} -> {len(x_hist_i) - 1:3d} iterações -> x* = {x_opt_i}")

grid0 = np.linspace(min(x0[0], c[0]) - 2, max(x0[0], c[0]) + 2, 60)
grid1 = np.linspace(min(x0[1], c[1]) - 2, max(x0[1], c[1]) + 2, 60)
X0, X1 = np.meshgrid(grid0, grid1)
Z = np.zeros_like(X0)
for i in range(X0.shape[0]):
    for j in range(X0.shape[1]):
        ponto = np.array([X0[i, j], X1[i, j]])
        Z[i, j] = J(ponto, A, b, c)

fig = go.Figure()
fig.add_trace(go.Surface(x=X0, y=X1, z=Z, colorscale="Viridis", opacity=0.6, showscale=False, name="J(x)"))

for a, cor in zip(alphas, cores):
    traj = resultados_alpha[a]
    Jz = np.array([J(p, A, b, c) for p in traj])
    fig.add_trace(
        go.Scatter3d(x=traj[:, 0], y=traj[:, 1], z=Jz, mode="lines+markers", line=dict(color=cor, width=4), marker=dict(size=3, color=cor), name=f"alpha={a}", hovertemplate="x0=%{x:.4f}<br>x1=%{y:.4f}<br>J(x)=%{z:.4f}<extra>alpha=" + str(a) + "</extra>",)
    )

fig.update_scenes(xaxis_title="x0", yaxis_title="x1", zaxis_title="J(x)")
fig.update_layout(title="Descida do gradiente sobre o paraboloide (x0, x1)", height=650, width=900, legend=dict(orientation="h", y=-0.05))
fig.show()


Iteração 0 : x = [4. 6.] -> J(x) = 20.000000
Iteração 1 : x = [5. 5.] -> J(x) = 0.000000
Iteração 2 : x = [5. 5.] -> J(x) = 0.000000 [Convergiu]

Ponto de mínimo encontrado: [5. 5.]
alpha = 0.025  ->  12 iterações -> x* = [4.99975586 5.00024414]
alpha = 0.05   ->   2 iterações -> x* = [5. 5.]
alpha = 0.075  ->  12 iterações -> x* = [4.99975586 5.00024414]


# Questão 3

In [87]:
import plotly.graph_objects as go
import numpy as np


def rastrigin(x, A=10):
    x1, x2 = x[0], x[1]
    return (2 * A + (x1**2 - A * np.cos(2 * np.pi * x1)) + (x2**2 - A * np.cos(2 * np.pi * x2)))


def gradiente_rastrigin(x, A):
    x1, x2 = x[0], x[1]
    dfdx1 = 2 * x1 + 2 * np.pi * A * np.sin(2 * np.pi * x1)
    dfdx2 = 2 * x2 + 2 * np.pi * A * np.sin(2 * np.pi * x2)
    return np.array([dfdx1, dfdx2])

def descida_gradiente(A, alpha, x0, max_iter=100, tol=1e-5):
    
    x = np.array(x0, dtype=float)
    x_hist = []
    hist = []
    Jx = rastrigin(x, A)
    hist.append(f"Iteração 0 : x = {x} -> J(x) = {Jx:.6f}\n")
    x_hist.append(x)

    for i in range(max_iter):
        grad = gradiente_rastrigin(x, A)
        xi = x - (alpha * grad)
        Jxi = rastrigin(xi, A)
        Jx = rastrigin(x, A)

        # Critério de convergência baseado na variação da função ou do ponto
        if np.linalg.norm(xi - x) <= tol or abs(Jxi - Jx) <= tol:
            hist.append(f"Iteração {i + 1} : x = {xi} -> J(x) = {Jxi:.6f} [Convergido]\n")
            x_hist.append(xi)
            return xi, hist, x_hist

        x = xi  
        hist.append(f"Iteração {i + 1} : x = {x} -> J(x) = {Jxi:.6f}\n")
        x_hist.append(xi)

    return x, hist, x_hist


A = 10
alpha = 0.005
x0 = {"A": np.array([0.5, 0.5]),"B": np.array([1.0, 1.0]),"C": np.array([-2.0, -2.0])}
cores_rastrigin = ["#1f77b4", "#ff7f0e", "#a31313"]
resultados_rastrigin = {}

for nome, x in x0.items():
    x_opt_i, hist_i, x_hist_i = descida_gradiente(A, alpha, x, max_iter=800, tol=1e-8)

    resultados_rastrigin[nome] = np.array(x_hist_i)
    print(f"{nome:<5} x0={x} -> {len(x_hist_i) - 1:3d} iterações -> x* = {x_opt_i} -> f(x*) = {rastrigin(x_opt_i):.6f}")

grid = np.linspace(-5, 5, 200)
X, Y = np.meshgrid(grid, grid)
Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = rastrigin([X[i, j], Y[i, j]])

fig = go.Figure()
fig.add_trace(
    go.Surface(x=X, y=Y, z=Z, colorscale="Viridis", opacity=0.85, showscale=False, name="f(x,y)")
)

for nome, cor in zip(x0, cores_rastrigin):
    traj = resultados_rastrigin[nome]
    Jz = np.array([rastrigin(p) for p in traj])
    fig.add_trace(
        go.Scatter3d(x=traj[:, 0],y=traj[:, 1],z=Jz,mode="lines+markers",line=dict(color=cor, width=5),marker=dict(size=2, color=cor),name=nome,hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>f=%{z:.4f}<extra>"+ nome+ "</extra>")
    )

fig.update_scenes(xaxis_title="x", yaxis_title="y", zaxis_title="f(x,y)")
fig.update_layout(title="Rastrigin", height=650, width=900,legend=dict(orientation="h", y=-0.05))
fig.show()

A     x0=[0.5 0.5] -> 386 iterações -> x* = [-2.74765318e-05 -2.74765318e-05] -> f(x*) = 0.000000
B     x0=[1. 1.] -> 304 iterações -> x* = [0.99498527 0.99498527] -> f(x*) = 1.989918
C     x0=[-2. -2.] -> 296 iterações -> x* = [-1.98993664 -1.98993664] -> f(x*) = 7.959663


In [63]:
import numpy as np
import plotly.graph_objects as go


def rastrigin(x, A=10):
    x1, x2 = x[0], x[1]
    return (
        2 * A
        + (x1**2 - A * np.cos(2 * np.pi * x1))
        + (x2**2 - A * np.cos(2 * np.pi * x2))
    )


def gradiente_rastrigin(x, A=10):
    x1, x2 = x[0], x[1]
    dfdx1 = 2 * x1 + 2 * np.pi * A * np.sin(2 * np.pi * x1)
    dfdx2 = 2 * x2 + 2 * np.pi * A * np.sin(2 * np.pi * x2)
    return np.array([dfdx1, dfdx2])


def descida_gradiente(A, alpha, x0, max_iter=100, tol=1e-5):
    x = np.array(x0, dtype=float)
    x_hist = [x.copy()]
    hist = []
    Jx = rastrigin(x, A)
    hist.append(f"Iteração 0 : x = {x} -> J(x) = {Jx:.6f}\n")

    for i in range(max_iter):
        grad = gradiente_rastrigin(x, A)
        xi = x - (alpha * grad)
        Jxi = rastrigin(xi, A)
        Jx = rastrigin(x, A)

        # Critério de convergência baseado na variação da função ou do ponto
        if np.linalg.norm(xi - x) <= tol or abs(Jxi - Jx) <= tol:
            hist.append(
                f"Iteração {i + 1} : x = {xi} -> J(x) = {Jxi:.6f} [Convergido]\n"
            )
            x_hist.append(xi.copy())
            return xi, hist, x_hist

        x = xi.copy()
        hist.append(f"Iteração {i + 1} : x = {x} -> J(x) = {Jxi:.6f}\n")
        x_hist.append(x.copy())

    return x, hist, x_hist


A = 10
x0 = np.array([0.3, 0.3])
alpha_nums = {"A": 0.0025, "B": 0.005, "C": 0.0075}
cores_rastrigin = ["#1f77b4", "#ff7f0e", "#a31313"]
resultados_rastrigin = {}

for nome, alpha in alpha_nums.items():
    x_opt_i, hist_i, x_hist_i = descida_gradiente(
        A, alpha, x0, max_iter=800, tol=1e-8
    )

    resultados_rastrigin[nome] = np.array(x_hist_i)
    # CORREÇÃO AQUI: trocado 'ponto' por 'x0'
    print(
        f"{nome:<5} alpha ={alpha} -> {len(x_hist_i) - 1:3d} iterações -> x* = {x_opt_i} -> f(x*) = {rastrigin(x_opt_i):.6f}"
    )

# Criação da malha 3D
grid = np.linspace(-5, 5, 200)
X, Y = np.meshgrid(grid, grid)
Z = np.zeros_like(X)
for i in range(X.shape[0]):
    for j in range(X.shape[1]):
        Z[i, j] = rastrigin([X[i, j], Y[i, j]])

fig = go.Figure()
fig.add_trace(
    go.Surface(
        x=X,
        y=Y,
        z=Z,
        colorscale="Viridis",
        opacity=0.85,
        showscale=False,
        name="f(x,y)",
    )
)

for nome, cor in zip(alpha_nums, cores_rastrigin):
    traj = resultados_rastrigin[nome]
    Jz = np.array([rastrigin(p) for p in traj])
    fig.add_trace(
        go.Scatter3d(
            x=traj[:, 0],
            y=traj[:, 1],
            z=Jz,
            mode="lines+markers",
            line=dict(color=cor, width=5),
            marker=dict(size=2, color=cor),
            name=nome,
            hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>f=%{z:.4f}<extra>"
            + nome
            + "</extra>",
        )
    )

fig.update_scenes(xaxis_title="x", yaxis_title="y", zaxis_title="f(x,y)")
fig.update_layout(
    title="Rastrigin", height=650, width=900, legend=dict(orientation="h", y=-0.05)
)
fig.show()

A     alpha =0.0025 ->   5 iterações -> x* = [1.56754197e-08 1.56754197e-08] -> f(x*) = 0.000000
B     alpha =0.005 -> 258 iterações -> x* = [2.7631151e-05 2.7631151e-05] -> f(x*) = 0.000000
C     alpha =0.0075 ->  59 iterações -> x* = [-0.23655312 -0.23655312] -> f(x*) = 18.424140
